# 15 Overview of measurements

<!-- navigation -->
[<kbd>← 12 Download river levels (Rijkswaterstaat)</kbd>](12%20Download%20river%20levels%20%28Rijkswaterstaat%29.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>20 Geohydrological scheme →</kbd>](20%20Geohydrological%20scheme.ipynb)

**Optional**: after the time series are imported (10, 11, 12)

Gives an overview of all time series per transect (period, time step, range, data availability, map), to check the data and choose the model period.

| | |
|---|---|
| **Input** | pastastore per transect |
| **Output** | overview table, figures in the transect folder 'timeseries' |

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt

import pandas as pd
print(f"timflow_levee_toolkit {tlt.__version__}")

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

In [ ]:
transects = project.transects()  # or a selection, e.g. ["Papendrecht West"]

## 1. Overview per transect

In [ ]:
rows = []
for transect in transects:
    pstore = project.open_pastastore(transect)
    names = pstore.oseries_names
    if not names:
        print(f"{transect}: the pastastore is empty")
        continue
    meta = project.transect_metadata(transect)
    for name in names:
        s = pstore.get_oseries(name).squeeze().dropna()
        rows.append({
            "transect": transect,
            "series": name,
            "kind": "river" if name == meta["loc_river"] else "piezometer",
            "start": s.index.min(),
            "end": s.index.max(),
            "values": len(s),
            "time_step_min": s.index.to_series().diff().median().total_seconds() / 60,
            "min": s.min(),
            "max": s.max(),
        })

    folder = project.transect_dir(transect, "timeseries")
    folder.mkdir(parents=True, exist_ok=True)
    title = f"{project.name} - {transect}"
    f, ax = tlt.plot_select(pstore, title=transect, subset="all series", figsize=(10, 6), units=units)
    tlt.finalize_and_save(f, folder, title, tag="series_all", file_label=transect)
    ax = pstore.plots.data_availability("oseries", set_yticks=True, figsize=(10, 4))
    tlt.finalize_and_save(ax.figure, folder, title, tag="data_availability", file_label=transect)
    try:
        ax = pstore.maps.oseries(labels=True, s=60, figsize=(8, 8))
        tlt.finalize_and_save(ax.figure, folder, title, tag="series_map", file_label=transect)
    except Exception as e:
        print(f"{transect}: no map ({type(e).__name__}: {e})")

overview = pd.DataFrame(rows).set_index(["transect", "series"])
overview.style.format(precision=2)

## 2. Detail: one transect in a chosen period

In [ ]:
transect_name = transects[0]
xlim = None  # the complete series, or a part, e.g. [pd.Timestamp("2025-12-03"), pd.Timestamp("2025-12-05")]

pstore = project.open_pastastore(transect_name)
tlt.plot_select(pstore, xlim=xlim, title=transect_name, subset="all series" if xlim is None else "selected period", figsize=(10, 6), units=units);

---

<!-- navigation -->
[<kbd>← 12 Download river levels (Rijkswaterstaat)</kbd>](12%20Download%20river%20levels%20%28Rijkswaterstaat%29.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>20 Geohydrological scheme →</kbd>](20%20Geohydrological%20scheme.ipynb)